# RAD-MOT: SAM 3 cho 2 video còn lại và video demo

Runtime: **GPU (T4)**. Ba phần độc lập sau bước chuẩn bị:

- **Phần 2** chạy SAM 3 trên `task_tunnel` và `task_night_far_objects` (YOLO detection, giống lần chạy `task_day_left_turn`), rồi so từng cửa sổ với VOS oracle.
- **Phần 3** (tuỳ chọn, tốn GPU) so SMP với RAD khi VOS là SAM 3.
- **Phần 4** làm video demo: ByteTrack | SMP | RAD | biểu đồ chi phí, có mask SAM 3.

Thời gian ước lượng ở chế độ `auto` trên T4: khoảng 3,8 giây cho mỗi khung SAM 3. Con số này suy ra từ thời điểm ghi file của lần chạy trước, nên chỉ là ước lượng.

| Việc | Khung SAM 3 | Thời gian |
|---|---|---|
| tunnel | ~450–560 | ~30–40 phút |
| night | ~2.400–3.000 | ~2,5–3 giờ |
| demo | ~300–700 | ~20–45 phút |

Nếu `fp16` nhanh hơn rõ ở bước 2a thì các con số này giảm theo.

## 1. Chuẩn bị

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json
CODE = '/content/drive/MyDrive/Saudh/computer_vision/rad-mot'
WORK = '/content/drive/MyDrive/Saudh/computer_vision/rad-work'
DS   = '/content/datasets/FastTracker'     # ổ tạm của phiên
DATA = f'{DS}/train'
SEQS_WITH_FRAMES = ['task_day_left_turn', 'task_tunnel', 'task_night_far_objects']
%cd $CODE
!pip -q install -r requirements.txt huggingface_hub
!python -m pytest -q

In [ ]:
# GT của cả 12 video (vài MB) + ảnh của 3 video cần dùng (mỗi video 0,5–1,6 GB)
!python scripts/download_fasttracker.py --out $DS --mode gt
need = [s for s in SEQS_WITH_FRAMES if not os.path.isdir(f'{DATA}/{s}/img1')]
print('cần tải ảnh:', need)
if need:
    !python scripts/download_fasttracker.py --out $DS --mode full --seqs {' '.join(need)}
!df -h /content | tail -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir('/content/sam3'):
    !git clone -q https://github.com/facebookresearch/sam3 /content/sam3
!pip -q install -e /content/sam3
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# detection YOLO đã lưu trên Drive từ lần trước; chỉ chạy lại nếu thiếu
missing = [s for s in SEQS_WITH_FRAMES if not os.path.exists(f'{WORK}/outputs/dets/{s}.txt')]
print('thiếu detection YOLO:', missing)
if missing:
    !pip -q install ultralytics
    !python scripts/detect_yolo.py --data $DATA --weights yolo11l.pt --imgsz 1280 --half --out $WORK/outputs/dets

## 2. SAM 3 cho 2 video còn lại

### 2a. Chọn precision (khoảng 2–4 phút)
So `ms/frame` và số khung *mask stays on its GT box* giữa hai chế độ. Chỉ chọn `fp16` khi nó nhanh hơn rõ **và** số khung bám đúng không giảm.

Lưu ý: `task_day_left_turn` đã chạy với `auto`. Nếu đổi sang `fp16` thì ghi rõ điều này trong báo cáo.

In [ ]:
!python scripts/smoke_sam3.py --data $DATA --seq task_tunnel --frame 100 --objects 5 --len 60 --precision auto
!python scripts/smoke_sam3.py --data $DATA --seq task_tunnel --frame 100 --objects 5 --len 60 --precision fp16

In [ ]:
PREC = 'auto'   # đổi thành 'fp16' nếu đạt điều kiện ở trên

### 2b. Pilot với SAM 3 (YOLO detection)
Mỗi video ghi kết quả vào Drive ngay khi chạy xong. Nếu phiên bị ngắt, chạy lại ô này: video nào đã có kết quả sẽ được bỏ qua.

Chạy `tunnel` trước vì nhanh; `night` tốn nhiều giờ GPU nhất.

In [ ]:
for s in ['task_tunnel', 'task_night_far_objects']:
    if os.path.exists(f'{WORK}/outputs/pilot_sam3/{s}/{s}/smp_windows_sam3.csv'):
        print('đã có', s); continue
    !python scripts/run_pilot.py --data $DATA --dets dir --det-dir $WORK/outputs/dets --backend sam3 --precision $PREC --seqs $s --out $WORK/outputs/pilot_sam3/$s

### 2c. So từng cửa sổ: oracle VOS và SAM 3
Hai lần chạy dùng cùng YOLO detection nên có cùng trigger, và mỗi cửa sổ được ghép theo `(track_id, entry_frame, witness_of)`.

Câu hỏi cần trả lời: kết luận đo bằng oracle có giữ nguyên khi thay bằng SAM 3 hay không.

Kết quả trên `task_day_left_turn`:
- 15/15 cửa sổ mà cả hai backend cùng chạy cho ra cùng quyết định SWAP hay không SWAP.
- Có 4 cửa sổ chỉ SAM 3 chạy được, vì oracle không bám được box YOLO không khớp với đối tượng GT nào.

In [ ]:
!python scripts/compare_backends.py --a $WORK/outputs/pilot_yolo --b $WORK/outputs/pilot_sam3 --out $WORK/outputs/compare_oracle_sam3

## 3. (Tuỳ chọn) SMP và RAD với SAM 3
Thiết lập này dùng oracle detection, giống thí nghiệm chính, và chỉ chạy 2 chính sách trên 3 video có ảnh. Mô hình rủi ro vẫn học out-of-fold trên cả 12 video.

Khối lượng theo sweep oracle là khoảng 5.700 khung SAM 3, tức nhiều giờ ở chế độ `auto`. Cửa sổ trùng giữa hai chính sách chỉ chạy SAM 3 một lần. Nên chạy phần này khi `fp16` nhanh, hoặc chỉ với `--sweep-seqs task_tunnel task_day_left_turn`.

In [ ]:
!python scripts/run_dispatch.py --data $DATA --dets oracle --backend sam3 --precision $PREC --policies smp:0.05 learned:1.0 --sweep-seqs task_tunnel task_day_left_turn task_night_far_objects --out $WORK/outputs/dispatch_sam3

## 4. Video demo
Script `scripts/make_demo.py` chạy theo thứ tự sau:

1. ByteTrack trên toàn video.
2. Mô hình rủi ro học trên **11 video còn lại**.
3. Tạo trigger của SMP và của RAD với cùng số lần kích hoạt.
4. Tự chọn đoạn ~300 khung quanh một lỗi đổi ID được sửa. Việc chọn dùng oracle nên rẻ; chọn theo ưu tiên: RAD sửa mà SMP không sửa, rồi RAD sửa, rồi SMP sửa.
5. Chỉ chạy SAM 3 cho các cửa sổ trong đoạn đó.

Kết quả gồm `demo.mp4` (1920×1080, 15 fps, tức nửa tốc độ), `keyframes/*.png` để chèn slide, `summary.json` và `events.csv`. Tất cả nằm trong `outputs/demo/<seq>_<đầu>-<cuối>_<backend>/`.

Muốn tự chọn đoạn thì thêm `--start 380 --frames 300`. Muốn xem toàn khung hình thay vì phóng to quanh sự kiện thì thêm `--crop none`.

In [ ]:
SEQ   = 'task_day_left_turn'    # ban ngày, dễ nhìn
DETS  = f'--dets dir --det-dir {WORK}/outputs/dets'      # YOLO (thực tế); '--dets oracle' để khớp thiết lập chính
TRAIN = f'{WORK}/outputs/dispatch/*/candidates.csv.gz'   # ứng viên đã gắn nhãn của lần chạy run_dispatch (oracle detection)
OUT   = f'{WORK}/outputs/demo'

### 4a. Xem trước bằng CPU (oracle VOS: vẽ hộp thay cho mask, khoảng 2–4 phút)

In [ ]:
!python scripts/make_demo.py --data $DATA --seq $SEQ $DETS --backend oracle --train-cands "$TRAIN" --out $OUT
from IPython.display import Image, Video, display
d = sorted(glob.glob(f'{OUT}/{SEQ}_*_oracle'), key=os.path.getmtime)[-1]
s = json.load(open(f'{d}/summary.json'))
print('đoạn:', s['clip'], '| sự kiện:', s['event'])
print(json.dumps(s['clip_eval'], indent=1))
for p in sorted(glob.glob(f'{d}/keyframes/*.png'))[:3]:
    display(Image(p, width=960))

### 4b. Demo thật với SAM 3 (cùng đoạn với bản xem trước vì cách chọn đoạn là tất định)

In [ ]:
!python scripts/make_demo.py --data $DATA --seq $SEQ $DETS --backend sam3 --precision $PREC --train-cands "$TRAIN" --out $OUT
d = sorted(glob.glob(f'{OUT}/{SEQ}_*_sam3'), key=os.path.getmtime)[-1]
print(json.dumps(json.load(open(f'{d}/summary.json'))['policies'], indent=1))
# bản nhỏ để xem trong notebook (bản đầy đủ 1920x1080 nằm trên Drive)
!ffmpeg -y -loglevel error -i "$d/demo.mp4" -vf scale=960:-2 -crf 30 /content/demo_preview.mp4
display(Video('/content/demo_preview.mp4', embed=True, width=960))

### Ghi chú khi trình bày
- ID đã sửa được hiển thị từ khung swap trở đi. Đây là hiệu chỉnh **offline** trong cửa sổ, giống SMP, không phải chạy thời gian thực.
- Panel RAD dùng mô hình học trên oracle detection của 11 video khác rồi áp dụng cho detection YOLO, nên có lệch phân phối giữa lúc học và lúc dùng. Chú thích dưới video có ghi điều này.
- Số lỗi đổi ID trên video chỉ tính trong đoạn clip, nên không so được với bảng kết quả toàn bộ.